# LangChain 메시지 히스토리 with Gemini

이 노트북은 Google Gemini API를 사용하여 LangChain에서 메시지 히스토리를 관리하는 방법을 보여줍니다.

## 주요 기능:
- 세션별 대화 기록 관리
- Gemini 2.0 Flash 모델 사용
- System Instruction 활용
- 스트리밍 응답 지원

## 1. 환경 설정 및 모델 초기화

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼 클래스
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI  # Google Gemini 모델을 사용하는 랭체인 클래스

In [ ]:
# 환경변수 로드
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("환경변수 GOOGLE_API_KEY가 설정되지 않았습니다.")

print("✅ 환경변수가 성공적으로 로드되었습니다.")

✅ 환경변수가 성공적으로 로드되었습니다.


In [16]:
# Gemini 모델 초기화 (system_instruction 사용)
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
    model_kwargs={
        "system_instruction": (
            "너는 사용자를 도와주는 상담사야. 공감적으로 답하고, "
            "모호하면 짧게 되물어봐. 필요하면 단계별로 안내해줘."
        )
    },
)

print("✅ Gemini 모델이 초기화되었습니다.")

✅ Gemini 모델이 초기화되었습니다.


## 2. 세션별 대화 기록 관리 설정

In [17]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

print("✅ 메시지 히스토리 관리 시스템이 설정되었습니다.")

✅ 메시지 히스토리 관리 시스템이 설정되었습니다.


## 3. 테스트 1: 첫 번째 세션에서 사용자 이름 기억하기

In [18]:
# 세션 abc2 설정
config = {"configurable": {"session_id": "abc2"}}

# 첫 번째 메시지: 자기소개
response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 이성용이야.")],
    config=config,
)

print("🤖 Gemini 응답:")
print(response.content)

🤖 Gemini 응답:
안녕하세요, 이성용 님. 무엇을 도와드릴까요?


## 4. 테스트 2: 같은 세션에서 이름 기억 확인

In [19]:
# 같은 세션에서 이름 질문
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,  # 같은 세션 ID 사용
)

print("🤖 Gemini 응답:")
print(response.content)

🤖 Gemini 응답:
이성용 님이시네요!


## 5. 테스트 3: 새로운 세션에서는 이름을 모름

In [20]:
# 새로운 세션 abc3 설정
config_new = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config_new,  # 새로운 세션 ID 사용
)

print("🤖 Gemini 응답 (새로운 세션):")
print(response.content)

🤖 Gemini 응답 (새로운 세션):
저는 챗봇이기 때문에 이름이 없습니다. 저는 사용자와 소통하고 정보를 제공하기 위해 개발된 인공지능 모델입니다.


## 6. 테스트 4: 첫 번째 세션으로 돌아가서 대화 기록 확인

In [21]:
# 다시 abc2 세션으로 돌아가기
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,  # 원래 세션 ID 사용
)

print("🤖 Gemini 응답 (원래 세션 복귀):")
print(response.content)

🤖 Gemini 응답 (원래 세션 복귀):
아까는 제게 본인 이름이 이성용이라고 말씀하셨어요. 혹시 다른 이야기를 하셨던가요? 기억이 잘 안 나시면 다시 말씀해주시면 됩니다. 😊


## 7. 테스트 5: 스트리밍 응답 테스트

In [23]:
print("🤖 Gemini 스트리밍 응답:")
print("="*50)

# 스트리밍으로 응답 받기
for r in with_message_history.stream(
    [HumanMessage(content="내가 어느 나라 사람인지 추측하고, 그 나라 문화 한 가지를 말해줘")],
    config=config,
):
    # r는 ChatMessage/AIMessage 조각일 수 있음 → content 이어 붙이기
    print(getattr(r, "content", ""), end="", flush=True)

print("\n" + "="*50)

🤖 Gemini 스트리밍 응답:
이름만으로는 어느 나라 사람인지 정확히 추측하기는 어렵습니다. '이성용'이라는 이름은 한국에서 흔히 사용되는 이름이기 때문에 한국 사람일 가능성이 높다고 추측할 수 있습니다.

한국 문화 중 하나를 말씀드리자면, **"존댓말 문화"**가 있습니다. 한국어는 나이, 사회적 지위, 친밀도 등에 따라 사용하는 언어가 달라집니다. 윗사람이나 처음 만나는 사람에게는 존댓말을 사용하는 것이 예의이며, 이는 상호 존중과 원활한 관계 유지를 위한 중요한 요소입니다. 이러한 존댓말 문화는 한국 사회의 위계질서를 반영하는 동시에, 상대방을 배려하는 마음을 표현하는 방식이라고 할 수 있습니다.



## 8. 세션 관리 및 상태 확인

In [24]:
# 현재 생성된 세션들 확인
print("📊 현재 활성 세션들:")
for session_id in store.keys():
    history = store[session_id]
    message_count = len(history.messages)
    print(f"  - 세션 {session_id}: {message_count}개 메시지")

print(f"\n📈 총 {len(store)} 개의 세션이 활성화되어 있습니다.")

📊 현재 활성 세션들:
  - 세션 abc2: 10개 메시지
  - 세션 abc3: 2개 메시지

📈 총 2 개의 세션이 활성화되어 있습니다.


In [25]:
# 특정 세션의 대화 기록 상세 조회
session_to_check = "abc2"
if session_to_check in store:
    print(f"💬 세션 '{session_to_check}'의 대화 기록:")
    print("-" * 40)
    
    for i, message in enumerate(store[session_to_check].messages, 1):
        speaker = "👤 사용자" if hasattr(message, 'content') and message.__class__.__name__ == "HumanMessage" else "🤖 AI"
        print(f"{i}. {speaker}: {message.content[:100]}{'...' if len(message.content) > 100 else ''}")
else:
    print(f"❌ 세션 '{session_to_check}'를 찾을 수 없습니다.")

💬 세션 'abc2'의 대화 기록:
----------------------------------------
1. 👤 사용자: 안녕? 난 이성용이야.
2. 🤖 AI: 안녕하세요, 이성용 님. 무엇을 도와드릴까요?
3. 👤 사용자: 내 이름이 뭐지?
4. 🤖 AI: 이성용 님이시네요!
5. 👤 사용자: 아까 우리가 무슨 얘기 했지?
6. 🤖 AI: 아까는 제게 본인 이름이 이성용이라고 말씀하셨어요. 혹시 다른 이야기를 하셨던가요? 기억이 잘 안 나시면 다시 말씀해주시면 됩니다. 😊
7. 👤 사용자: 내가 어느 나라 사람인지 추측하고, 그 나라 문화 한 가지를 말해줘
8. 🤖 AI: 이름만으로는 어느 나라 사람인지 정확히 추측하기는 어렵습니다. '이성용'이라는 이름은 한국에서 흔히 사용되는 이름이기 때문에 한국 사람일 가능성이 높다고 추측할 수 있습니다.

한...
9. 👤 사용자: 내가 어느 나라 사람인지 추측하고, 그 나라 문화 한 가지를 말해줘
10. 🤖 AI: 이름만으로는 어느 나라 사람인지 정확히 추측하기는 어렵습니다. '이성용'이라는 이름은 한국에서 흔히 사용되는 이름이기 때문에 한국 사람일 가능성이 높다고 추측할 수 있습니다.

한...


## 9. 추가 기능: 새로운 대화 시도

In [26]:
# 새로운 주제로 대화하기
config = {"configurable": {"session_id": "abc2"}}  # 기존 세션 계속 사용

response = with_message_history.invoke(
    [HumanMessage(content="오늘 날씨가 좋다면 뭘 하면 좋을까?")],
    config=config,
)

print("🤖 Gemini 응답:")
print(response.content)

🤖 Gemini 응답:
오늘 날씨가 좋다면 정말 할 수 있는 게 많죠! 몇 가지 아이디어를 드릴게요:

**야외 활동:**

*   **피크닉:** 돗자리, 간단한 음식, 책 등을 챙겨 가까운 공원이나 강가로 피크닉을 떠나세요. 햇볕을 쬐면서 여유로운 시간을 보내는 건 정말 행복하죠.
*   **산책/하이킹:** 맑은 공기를 마시며 가볍게 산책하거나, 조금 더 활동적인 하이킹을 즐겨보세요. 자연 속에서 몸과 마음을 재충전할 수 있습니다.
*   **자전거 타기:** 자전거를 타고 동네를 돌아다니거나, 자전거 도로를 따라 시원하게 달려보세요. 운동도 되고 기분 전환도 됩니다.
*   **야외 스포츠:** 친구들과 함께 공원에서 축구, 농구, 배드민턴 등을 즐겨보세요. 활동적인 시간을 보내며 스트레스도 날릴 수 있습니다.
*   **캠핑/글램핑:** 자연 속에서 하룻밤을 보내는 캠핑이나 글램핑은 특별한 경험이 될 거예요. 맛있는 음식을 해 먹고, 밤하늘의 별을 보며 힐링하세요.

**문화/예술 활동:**

*   **야외 박물관/미술관:** 날씨 좋은 날 야외에 있는 박물관이나 미술관을 방문하여 예술 작품을 감상해보세요.
*   **공연/축제:** 야외에서 열리는 공연이나 축제에 참여하여 다양한 볼거리를 즐겨보세요.

**휴식/여유:**

*   **카페/레스토랑 테라스:** 햇볕이 잘 드는 테라스가 있는 카페나 레스토랑에서 맛있는 음료나 음식을 즐기며 여유로운 시간을 보내세요.
*   **독서:** 공원 벤치나 집 앞 마당에 앉아 좋아하는 책을 읽어보세요. 따뜻한 햇살 아래에서 책을 읽는 건 정말 낭만적이죠.

어떤 활동을 좋아하시는지에 따라 더 구체적인 추천을 해 드릴 수 있습니다. 평소에 어떤 활동을 즐겨 하시나요?


In [27]:
# 이전 대화와 연결되는지 확인
response = with_message_history.invoke(
    [HumanMessage(content="아까 내 이름과 함께 추천해줄 수 있어?")],
    config=config,
)

print("🤖 Gemini 응답 (이름 기억 + 맥락 연결):")
print(response.content)

🤖 Gemini 응답 (이름 기억 + 맥락 연결):
아, 물론이죠! 이성용 님께 날씨 좋은 날 추천하는 활동입니다.

**이성용 님께 추천하는 날씨 좋은 날 활동:**

*   **한강 공원 피크닉:** 이성용 님, 돗자리와 간단한 간식을 챙겨 한강 공원으로 피크닉을 떠나보세요. 따뜻한 햇볕 아래에서 맛있는 음식을 먹고, 한강을 바라보며 여유로운 시간을 보내는 건 어떠세요? 특히, 저녁 노을이 질 때쯤 가면 더욱 멋진 풍경을 감상할 수 있을 거예요.

*   **북악스카이웨이 드라이브 & 북악정 팔각정 전망:** 이성용 님, 시원한 바람을 맞으며 북악스카이웨이를 드라이브하는 건 어떠세요? 북악정 팔각정에 올라 서울 시내를 한눈에 내려다보는 멋진 경험을 하실 수 있을 거예요. 드라이브 후에는 근처 분위기 좋은 카페에서 커피 한 잔을 즐기는 것도 좋겠네요.

*   **올림픽공원 산책 & 조각공원 관람:** 이성용 님, 올림픽공원에서 산책하며 아름다운 자연을 만끽해보세요. 특히 조각공원에는 다양한 조각 작품들이 전시되어 있어 예술적인 감성을 충전할 수 있습니다. 산책 후에는 공원 내 카페에서 시원한 음료를 마시며 휴식을 취하는 것도 추천합니다.

이성용 님은 어떤 활동에 가장 끌리시나요? 아니면 다른 취향이 있으시다면, 더 맞춤형으로 추천해 드릴 수 있습니다! 😊


## 10. 정리 및 요약

이 노트북에서 확인한 내용:

✅ **세션별 메시지 히스토리 관리**
- 세션 ID를 통한 대화 기록 분리
- 같은 세션에서는 이전 대화 내용 기억
- 다른 세션에서는 독립적인 대화 시작

✅ **Gemini 2.0 Flash 모델 활용**
- System Instruction으로 AI 성격 설정
- 한국어 대화 지원
- 실시간 스트리밍 응답

✅ **LangChain 통합**
- `RunnableWithMessageHistory`로 히스토리 관리
- `InMemoryChatMessageHistory`로 메모리 저장
- 다양한 메시지 타입 지원

In [28]:
# 최종 세션 상태 출력
print("🎉 LangChain 메시지 히스토리 with Gemini 테스트 완료!")
print(f"📊 최종 통계:")
print(f"  - 활성 세션 수: {len(store)}")
print(f"  - 총 메시지 수: {sum(len(history.messages) for history in store.values())}")
print("")
print("💡 주요 학습 내용:")
print("  1. Google Gemini API와 LangChain 통합")
print("  2. 세션별 대화 기록 관리")
print("  3. System Instruction 활용")
print("  4. 스트리밍 응답 처리")
print("  5. 메시지 히스토리 상태 모니터링")

🎉 LangChain 메시지 히스토리 with Gemini 테스트 완료!
📊 최종 통계:
  - 활성 세션 수: 2
  - 총 메시지 수: 16

💡 주요 학습 내용:
  1. Google Gemini API와 LangChain 통합
  2. 세션별 대화 기록 관리
  3. System Instruction 활용
  4. 스트리밍 응답 처리
  5. 메시지 히스토리 상태 모니터링
